# Ноутбук с препроцессингом, обучением и сравнением моделей

## 1. Импорт бибилиотек и конфигурация проекта

In [30]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.linear_model import Ridge
from category_encoders.cat_boost import CatBoostEncoder
from sklearn.model_selection import cross_val_score
import pyarrow
import phik
from phik.report import plot_correlation_matrix
import mlflow
from datetime import datetime
import category_encoders as ce

In [24]:
# Создадим словарь конфигураций.

CONFIG = {
    # Константы
    "DEV_MODE": False,
    "DEV_SAMPLE_SIZE": 100000,
    "RANDOM_STATE": 42,
    # Целевая переменная 
    "TARGET": "Цена",
    "YEAR": datetime.now().year
}

In [25]:
train = pd.read_parquet("../data/features/train_features.parquet")
test = pd.read_parquet("../data/features/test_features.parquet")

In [26]:
y_train = train[CONFIG["TARGET"]]
X_train = train.drop(columns=[CONFIG["TARGET"]])

y_test = test[CONFIG["TARGET"]]
X_test = test.drop(columns=[CONFIG["TARGET"]])

## 2. Пайплайн предобработки

In [27]:
num_cols = X_train.select_dtypes('number').columns.to_list()
low_card_cols = ['Тип двигателя', 'Коробка передач', 'Привод', 'Руль', 'Цвет', 'Тип кузова']
high_card_cols = ['Марка', 'Регион', 'Модель']

Переводим колонки тпиа object в category для моделей.

In [ ]:
cat_cols = low_card_cols + high_card_cols
X_train[cat_cols] = X_train[cat_cols].astype('category')
X_test[cat_cols] = X_test[cat_cols].astype('category')

In [31]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', RobustScaler(), num_cols),
        ('low_card', OneHotEncoder(handle_unknown='ignore', sparse_output=False), low_card_cols),
        ('high_card', CatBoostEncoder(handle_unknown='value'), high_card_cols)
    ]
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Ridge())
])

In [29]:
scores = cross_val_score(
    pipeline,
    X_train,
    y_train,
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1
)
print(f"Средний MAE линейной регрессии на кросс-валидации: {-scores.mean():.2f}")

Средний MAE линейной регрессии на кросс-валидации: 0.24
